<a href="https://colab.research.google.com/github/Sunil-143-as/Practical-1/blob/main/PRACTICAL_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# --------------------------------------------------------------
# STEP 1: Install Required Software
# --------------------------------------------------------------
import subprocess
import sys
import os
import pandas as pd
print("=" * 70)
print(" PACKET ANALYSIS USING WIRESHARK / TSHARK")
print("=" * 70)
print("\n[1] Installing TShark...")
install_tshark = subprocess.run(
[
"bash",
"-c",
"apt-get update -qq && "
"DEBIAN_FRONTEND=noninteractive "
"apt-get install -y tshark -qq"
],
capture_output=True,
text=True
)
if install_tshark.returncode == 0:
    print("TShark installation completed.")
else:
    print("TShark installation returned an error.")
    print(install_tshark.stderr)
# --------------------------------------------------------------
# STEP 2: Install Scapy
# --------------------------------------------------------------
print("\n[2] Installing Scapy...")
install_scapy = subprocess.run(
[sys.executable, "-m", "pip", "install", "-q", "scapy"],
capture_output=True,
text=True
)
if install_scapy.returncode == 0:
    print("Scapy installation completed.")
else:
    print("Scapy installation error:")
    print(install_scapy.stderr)
# --------------------------------------------------------------
# STEP 3: Import Scapy
# --------------------------------------------------------------
from scapy.all import (
IP,
IPv6,
TCP,
UDP,
ICMP,
DNS,
DNSQR,
Raw,
Ether,
wrpcap,
rdpcap
)
print("\n[3] Scapy imported successfully.")
# --------------------------------------------------------------
# STEP 4: Check TShark Version
# --------------------------------------------------------------
print("\n[4] Checking TShark installation...")
version = subprocess.run(
["tshark", "--version"],
capture_output=True,
text=True
)
if version.returncode == 0:
    print(version.stdout.splitlines()[0])
else:
    print("Unable to execute TShark.")
    print(version.stderr)
# --------------------------------------------------------------
# STEP 5: Define PCAP File
# --------------------------------------------------------------
pcap_file = "/content/packet_analysis_capture.pcap"
print("\n[5] PCAP File:")
print(pcap_file)
# --------------------------------------------------------------
# STEP 6: Create Controlled Packets
# --------------------------------------------------------------
print("\n[6] Generating controlled educational packets...")
packets = []
# --------------------------------------------------------------
# ICMP PACKETS
# --------------------------------------------------------------
icmp_request = (
IP(src="192.168.1.10", dst="8.8.8.8") /
ICMP(type=8) /
Raw(load="Educational ICMP Echo Request")
)
icmp_reply = (
IP(src="8.8.8.8", dst="192.168.1.10") /
ICMP(type=0) /
Raw(load="Educational ICMP Echo Reply")
)
packets.append(icmp_request)
packets.append(icmp_reply)
# --------------------------------------------------------------
# TCP PACKETS
# --------------------------------------------------------------
tcp_syn = (
IP(src="192.168.1.10", dst="192.168.1.20") /
TCP(sport=50000, dport=80, flags="S")
)
tcp_syn_ack = (
IP(src="192.168.1.20", dst="192.168.1.10") /
TCP(sport=80, dport=50000, flags="SA")
)
tcp_ack = (
IP(src="192.168.1.10", dst="192.168.1.20") /
TCP(sport=50000, dport=80, flags="A")
)
packets.append(tcp_syn)
packets.append(tcp_syn_ack)
packets.append(tcp_ack)
# --------------------------------------------------------------
# UDP PACKET
# --------------------------------------------------------------
udp_packet = (
IP(src="192.168.1.10", dst="192.168.1.30") /
UDP(sport=40000, dport=5000) /
Raw(load="Educational UDP Data")
)
packets.append(udp_packet)
# --------------------------------------------------------------
# DNS QUERY PACKET
# --------------------------------------------------------------
dns_packet = (
IP(src="192.168.1.10", dst="8.8.8.8") /
UDP(sport=53000, dport=53) /
DNS(
rd=1,
qd=DNSQR(qname="example.com")
)
)
packets.append(dns_packet)
# --------------------------------------------------------------
# HTTP-STYLE TCP PACKET
# --------------------------------------------------------------
http_packet = (
IP(src="192.168.1.10", dst="192.168.1.20") /
TCP(sport=51000, dport=80, flags="PA") /
Raw(
load="GET / HTTP/1.1\r\n"
"Host: example.com\r\n"
"Connection: close\r\n\r\n"
)
)
packets.append(http_packet)
# --------------------------------------------------------------
# ARP PACKETS
# --------------------------------------------------------------
from scapy.all import ARP
arp_request = ARP(
op=1,
psrc="192.168.1.10",
pdst="192.168.1.1",
hwsrc="02:00:00:00:00:10"
)
arp_reply = ARP(
op=2,
psrc="192.168.1.1",
pdst="192.168.1.10",
hwsrc="02:00:00:00:00:01",
hwdst="02:00:00:00:00:10"
)
packets.append(arp_request)
packets.append(arp_reply)
# --------------------------------------------------------------
# STEP 7: Save Packets to PCAP
# --------------------------------------------------------------
print("\n[7] Saving packets to PCAP file...")
wrpcap(pcap_file, packets)
print("Packets saved successfully.")
print("Number of packets generated:", len(packets))
# --------------------------------------------------------------
# STEP 8: Verify PCAP File
# --------------------------------------------------------------
print("\n[8] Checking PCAP file...")
if os.path.exists(pcap_file):
    file_size = os.path.getsize(pcap_file)
    print("PCAP file exists.")
    print("File size:", file_size, "bytes")
else:
    print("PCAP file was not created.")
# --------------------------------------------------------------
# STEP 9: Read PCAP Using Scapy
# --------------------------------------------------------------
print("\n[9] Reading packets using Scapy...")
captured_packets = rdpcap(pcap_file)
print("Packets read:", len(captured_packets))
# --------------------------------------------------------------
# STEP 10: Display Basic Packet Information
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" PACKET INFORMATION")
print("=" * 70)
for i, packet in enumerate(captured_packets, start=1):
    print("\nPacket Number:", i)
    if IP in packet:
        print("Source IP :", packet[IP].src)
        print("Destination IP :", packet[IP].dst)
    elif ARP in packet:
        print("Source IP :", packet[ARP].psrc)
        print("Destination IP :", packet[ARP].pdst)
    print("Protocol :", packet.summary())
# --------------------------------------------------------------
# STEP 11: Analyze All Packets Using TShark
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" TSHARK PACKET ANALYSIS")
print("=" * 70)
tshark_all = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,"
,
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"_ws.col.Protocol",
"-e",
"frame.len"
],
capture_output=True,
text=True
)
print(tshark_all.stdout)
# --------------------------------------------------------------
# STEP 12: Analyze TCP Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" TCP PACKET ANALYSIS")
print("=" * 70)
tcp_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"tcp",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"tcp.srcport",
"-e",
"tcp.dstport",
"-e",
"tcp.flags.str"
],
capture_output=True,
text=True
)
print(tcp_analysis.stdout)
# --------------------------------------------------------------
# STEP 13: Analyze UDP Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" UDP PACKET ANALYSIS")
print("=" * 70)
udp_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"udp",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"udp.srcport",
"-e",
"udp.dstport"
],
capture_output=True,
text=True
)
print(udp_analysis.stdout)
# --------------------------------------------------------------
# STEP 14: Analyze ICMP Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" ICMP PACKET ANALYSIS")
print("=" * 70)
icmp_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"icmp",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"icmp.type",
"-e",
"icmp.code"
],
capture_output=True,
text=True
)
print(icmp_analysis.stdout)
# --------------------------------------------------------------
# STEP 15: Analyze DNS Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" DNS PACKET ANALYSIS")
print("=" * 70)
dns_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"dns",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"dns.qry.name",
"-e",
"dns.qry.type"
],
capture_output=True,
text=True
)
print(dns_analysis.stdout)
# --------------------------------------------------------------
# STEP 16: Analyze ARP Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" ARP PACKET ANALYSIS")
print("=" * 70)
arp_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"arp",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"arp.src.proto_ipv4",
"-e",
"arp.dst.proto_ipv4",
"-e",
"arp.opcode"
],
capture_output=True,
text=True
)
print(arp_analysis.stdout)
# --------------------------------------------------------------
# STEP 17: Analyze HTTP Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" HTTP PACKET ANALYSIS")
print("=" * 70)
http_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"http",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"http.request.method",
"-e",
"http.host",
"-e",
"http.request.uri"
],
capture_output=True,
text=True
)
print(http_analysis.stdout)
# --------------------------------------------------------------
# STEP 18: Identify TCP SYN Packets
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" TCP SYN ANALYSIS")
print("=" * 70)
syn_analysis = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-Y",
"tcp.flags.syn == 1",
"-T",
"fields",
"-E",
"header=y",
"-E",
"separator=,",
"-e",
"frame.number",
"-e",
"ip.src",
"-e",
"ip.dst",
"-e",
"tcp.srcport",
"-e",
"tcp.dstport",
"-e",
"tcp.flags.str"
],
capture_output=True,
text=True
)
print(syn_analysis.stdout)
# --------------------------------------------------------------
# STEP 19: Protocol Statistics
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" PROTOCOL STATISTICS")
print("=" * 70)
protocol_stats = subprocess.run(
[
"tshark",
"-r",
pcap_file,
"-q",
"-z",
"io,phs"
],
capture_output=True,
text=True
)
print(protocol_stats.stdout)
# --------------------------------------------------------------
# STEP 20: Packet Count
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" PACKET SUMMARY")
print("=" * 70)
print("Total Packets Generated :", len(packets))
print("PCAP File :", pcap_file)
print("PCAP Size :", os.path.getsize(pcap_file), "bytes")
# --------------------------------------------------------------
# STEP 21: Completion Message
# --------------------------------------------------------------
print("\n" + "=" * 70)
print(" PACKET ANALYSIS COMPLETED")
print("=" * 70)
print("\nStudents should record the actual TShark output")
print("and complete the observation tables.")

 PACKET ANALYSIS USING WIRESHARK / TSHARK

[1] Installing TShark...
TShark installation completed.

[2] Installing Scapy...
Scapy installation completed.

[3] Scapy imported successfully.

[4] Checking TShark installation...
TShark (Wireshark) 4.2.2 (Git v4.2.2 packaged as 4.2.2-1.1build3).

[5] PCAP File:
/content/packet_analysis_capture.pcap

[6] Generating controlled educational packets...

[7] Saving packets to PCAP file...


Packets saved successfully.
Number of packets generated: 10

[8] Checking PCAP file...
PCAP file exists.
File size: 673 bytes

[9] Reading packets using Scapy...
Packets read: 10

 PACKET INFORMATION

Packet Number: 1
Source IP : 192.168.1.10
Destination IP : 8.8.8.8
Protocol : IP / ICMP 192.168.1.10 > 8.8.8.8 echo-request 0 / Raw

Packet Number: 2
Source IP : 8.8.8.8
Destination IP : 192.168.1.10
Protocol : IP / ICMP 8.8.8.8 > 192.168.1.10 echo-reply 0 / Raw

Packet Number: 3
Source IP : 192.168.1.10
Destination IP : 192.168.1.20
Protocol : IP / TCP 192.168.1.10:50000 > 192.168.1.20:http S

Packet Number: 4
Source IP : 192.168.1.20
Destination IP : 192.168.1.10
Protocol : IP / TCP 192.168.1.20:http > 192.168.1.10:50000 SA

Packet Number: 5
Source IP : 192.168.1.10
Destination IP : 192.168.1.20
Protocol : IP / TCP 192.168.1.10:50000 > 192.168.1.20:http A

Packet Number: 6
Source IP : 192.168.1.10
Destination IP : 192.168.1.30
Protocol : IP / UDP 192.168.1.10:40000 > 192.168.1.30:5000 /